In [ ]:
from models import load_model_only
from utils import *

config = parse_config('../configs/tversky_sirl.yaml')
model = load_model_only(config, "../results/tversky_sirl_gridrobot/tversky_sirl_dim10_fbank4_seed0.pth") # wtf lol why am i returning ckpt path

# TODO eventually write a script to loop all *tversky* ckpts

In [6]:
data = load_data(config)

loading data: gridrobot_1960


In [12]:
all_trajs = data["trajs"]
all_feats = data["features"]

In [15]:
all_trajs.shape

(1960, 19)

In [14]:
all_feats.shape

(1960, 2)

In [ ]:
model

TverskySIRL(
  (encoder): Sequential(
    (0): Linear(in_features=19, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
  (tversky_sim): TverskySimilarity(
    (feature_bank): Embedding(4, 10)
  )
)

In [ ]:
feature_bank = model.tversky_sim.feature_bank.weight.detach()
feature_bank.shape # 4 Tversky features, of 10-dim embeddings

torch.Size([4, 10])

In [5]:
# from tversky/test_mnist.py
# edited
import torch.nn.functional as F
def compute_salience(x, feature_bank):
    """
    x is (N, D)
    feature_bank is (F, D)
    """
    feature_measures = x @ feature_bank.T # (N, F)
    salience_measures = F.relu(feature_measures).sum(-1)
    return salience_measures

In [ ]:
all_embeds = model(torch.as_tensor(all_trajs, dtype=torch.float32))
all_salience = compute_salience(all_embeds, feature_bank).detach()

In [25]:
feature_bank

tensor([[0.8367, 0.5799, 0.8378, 0.3441, 0.4684, 0.8598, 0.3034, 0.3625, 0.3533,
         0.1264],
        [0.8044, 0.5495, 0.5488, 0.7720, 0.1128, 0.0539, 0.0962, 0.6704, 0.3359,
         0.8343],
        [0.8065, 0.1609, 0.5367, 0.1851, 0.7149, 0.5249, 0.6147, 0.5522, 0.5272,
         0.0997],
        [0.4405, 0.7476, 0.2013, 0.9624, 0.5613, 0.3369, 0.4496, 0.2895, 0.5051,
         0.3690]])

In [24]:
all_embeds

tensor([[-1.2614,  0.5069, -0.0915,  ..., -0.8840, -1.0233, -1.1913],
        [-0.9438, -0.3372, -0.4162,  ..., -0.8598, -0.9027, -1.2197],
        [-1.2052, -0.0211, -0.4859,  ..., -0.8060, -0.8253, -1.3171],
        ...,
        [-1.0450, -0.0975, -0.2767,  ..., -0.8559, -1.0816, -1.3079],
        [-0.9513, -0.0738, -0.2463,  ..., -0.8201, -1.0495, -1.1741],
        [-0.9928, -0.3276, -0.2757,  ..., -0.8205, -0.7894, -1.2292]],
       grad_fn=<AddmmBackward0>)

In [23]:
all_salience

tensor([0., 0., 0.,  ..., 0., 0., 0.])

all the saliences are 0... the features are all positive, so the embeddings must be the problem? let's try centering...

In [26]:
mu = all_embeds.mean(0)        # shape [6]
centered_embeds = all_embeds - mu     # [10000, 6]

In [27]:
centered_salience = compute_salience(centered_embeds, feature_bank).detach()

In [28]:
torch.max(centered_salience)

tensor(8.1493)

In [29]:

torch.min(centered_salience)

tensor(0.)

In [32]:
min_salience_idx = np.argmin(centered_salience).item()
min_salience_traj = all_trajs[min_salience_idx]
max_salience_idx = np.argmax(centered_salience).item()
max_salience_traj = all_trajs[max_salience_idx]
print(f"min salience traj has features {all_feats[min_salience_idx]}")
print(f"max salience traj has features {all_feats[max_salience_idx]}")

min salience traj has features [0.09050898 1.        ]
max salience traj has features [0.59050898 1.        ]


| `features` | `(N, 2)` | Per-trajectory features `phi` (scaled): `computer_dist` = summed distance from each waypoint to the grid center; `joint_up` = magnitude of the joint angle. |

In [ ]:
# TODO function to visualize a gridrobot trajectory would be great
import sys
sys.path.insert(0, '../../../simulated_data/001-gridrobot/')
from gridrobot import Gridrobot
    
config = {
    "X": 5,
    "Y": 5,
    "obstacles": [],
    "starts": [[0, 0], [0, 4], [4, 0], [4, 4]],
    "goals": [[4, 4], [4, 0], [0, 4], [0, 0]],
    "features": ["computer_dist", "joint_up"],
    "thetas": [[ -10.0, -10.0 ],
              [ 0.0, -10.0 ],
              [ 10.0, -10.0 ],
              [ -10.0, 0.0 ],
              [ 10.0, 0.0 ],
              [ -10.0, 10.0 ],
              [ 0.0, 10.0 ],
              [ 10.0, 10.0 ]
    ],
    "beta": 10.0,
    "feature_scaling": "normalize",
    "train_test_split": 0.8
}
env = Gridrobot(config["X"], config["Y"], config["obstacles"], config["starts"], config["goals"])

------ Agent MDP ------
num X :  5
num Y :  5
obstacles :  0
-----------------------


In [34]:
env.visualize_one_traj(min_salience_traj)

AttributeError: 'Gridrobot' object has no attribute 'visualize_one_traj'